# 04 – Model Evaluation & Business Interpretation

**Goal:** Evaluate the final tuned classifier on the held-out test set, analyze errors and decision thresholds, and interpret the K-Means customer segments for business action.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import train_test_split

from ml import config, train

## 1. Held-Out Test Set Metrics & Classification Report

In [ ]:
bundle = joblib.load(config.MODEL_PATH)
pipe = bundle["pipeline"]
report = json.loads(config.METRICS_PATH.read_text())
print("Best Model:", report["best_model"], "| Best Params:", report["best_params"])
pd.Series(report["test_metrics"])

In [ ]:
df = train.load_dataset()
X, y, _ = train.prepare_xy(df)
_, X_test, _, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
labels = [config.NEGATIVE_CLASS, config.POSITIVE_CLASS]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, display_labels=labels, cmap="Greens", ax=axes[0])
RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="grey")
plt.tight_layout(); plt.show()
print(classification_report(y_test, pipe.predict(X_test), target_names=labels))

## 2. Feature Importance & Customer Segmentation Summary

In [ ]:
pd.Series(report["feature_importance"]).sort_values().plot.barh(figsize=(7, 4), color="#2F855A")
plt.title("Feature Importance (Final Classification Model)")
plt.tight_layout(); plt.show()

seg = json.loads(config.CLUSTERING_METRICS_PATH.read_text())
print(f"K-Means (K={seg['n_clusters']}) Silhouette Score: {seg['silhouette_score']}")
pd.DataFrame(seg["profiles"])[["cluster_id", "segment_name", "size", "share_pct", "conversion_rate_pct", "recommended_action"]]